Lab 1: Basic Neural Networks

This notebook concerns the neural network unit and focuses on how a network is trained.

## Learning goals

By the end of this lab, you should be able to:

1. Explain how a neural network classifier is trained by minimizing a loss function.
2. Describe the role of backpropagation in updating weights.
3. Implement a training loop in PyTorch for binary classification.
4. Compare a neural network with logistic regression on the same dataset.


## 1. Setup

We begin by importing the packages used in this lab and setting a random seed.



In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)


## 2. Load dataset

We will continue using the Wisconsin Breast Cancer dataset from `scikit-learn`.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Feature matrix shape:", X.shape)
print("Outcome shape:", y.shape)
print()
print("Class counts:")
print(y.value_counts().sort_index())

## 3. Train, validation, and test split

As in earlier modules, we separate the data into training, validation, and test sets.

- The training set is used to estimate model parameters.
- The validation set is used for model comparison and tuning.
- The test set is used only for final evaluation.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=SEED
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

## 4. Standardize the predictors

Neural networks are usually easier to train when predictors are on comparable scales.

We fit the scaler on the training set only, then apply it to the validation and test sets.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training mean, first 5 columns:", X_train_scaled.mean(axis=0)[:5].round(4))
print("Training sd, first 5 columns:", X_train_scaled.std(axis=0)[:5].round(4))

## 5. Convert the data to PyTorch tensors

PyTorch works with tensors rather than pandas objects or NumPy arrays.
For binary classification with `BCEWithLogitsLoss`, it is convenient to store the outcome as a float tensor with shape $(n, 1)$.

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy().reshape(-1, 1), dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.to_numpy().reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy().reshape(-1, 1), dtype=torch.float32)

print(X_train_tensor.shape, y_train_tensor.shape)
print(X_val_tensor.shape, y_val_tensor.shape)
print(X_test_tensor.shape, y_test_tensor.shape)

## 6. A quick review of the binary classification loss

For one observation, binary cross-entropy loss is

$
\ell_i = -\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right].
$

For $n$ observations, the average loss is

$
L = -\frac{1}{n} \sum_{i=1}^{n} \left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right].
$

In PyTorch, we will use `BCEWithLogitsLoss`, which combines:

1. the linear output of the network
2. the sigmoid transformation
3. the binary cross-entropy loss

This is numerically more stable than applying the sigmoid separately and then computing the loss.

## 7. Define a simple feedforward neural network

We begin with one hidden layer:

$
\text{input} \rightarrow \text{hidden layer} \rightarrow \text{output}.
$

The hidden layer uses the ReLU activation function. The output layer returns a single logit.

In [ ]:
class BasicNN(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 1)
        )

    def forward(self, x):
        return self.network(x)

## 8. Choose a batch size

We will train the network using mini-batches (required in Stochastic Gradient Descent). The batch size is set here, while the `DataLoader` itself will be created inside the training function so that each training run starts from the same random state.


In [ ]:
batch_size = 32


## 9. Write a training function

The function below trains a network and stores the training and validation loss over epochs.
It also returns the fitted model.

We include an optional `weight_decay` argument. In gradient-based optimization, this is an $L_2$ penalty and plays a role similar to ridge regularization.

In [ ]:

def train_model(
    hidden_units=16,
    lr=0.01,
    epochs=200,
    weight_decay=0.0,
    seed=SEED
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    model = BasicNN(input_dim=X_train_tensor.shape[1], hidden_units=hidden_units)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_loss = loss_fn(val_logits, y_val_tensor).item()

        history["epoch"].append(epoch)
        history["train_loss"].append(np.mean(batch_losses))
        history["val_loss"].append(val_loss)

    history = pd.DataFrame(history)
    return model, history


## 10. Train a first neural network

We start with:

- one hidden layer
- 16 hidden units
- learning rate $0.01$
- no weight decay

In [ ]:
model_16, history_16 = train_model(
    hidden_units=16,
    lr=0.01,
    epochs=200,
    weight_decay=0.0
)

history_16.head()

## 11. Plot the training and validation loss

A useful first diagnostic is whether the loss decreases over time and whether the validation loss begins to flatten or increase.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_16["epoch"], history_16["train_loss"], label="training loss")
plt.plot(history_16["epoch"], history_16["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

The network fits the training data extremely well, but the validation loss starts to rise after the early epochs. This pattern indicates overfitting. In practice, we would not choose the final epoch automatically. Instead, we would use the validation set to decide when to stop training or how much regularization to apply.

## 12. Use the validation set to choose the training epoch

One simple strategy is to choose the epoch with the smallest validation loss. This is closely related to early stopping.


In [ ]:
best_epoch = int(history_16.loc[history_16["val_loss"].idxmin(), "epoch"])
best_val_loss = history_16["val_loss"].min()

print("Best epoch based on validation loss:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))

## 13. Generate predicted probabilities

The network returns logits. To convert logits to probabilities, we apply the sigmoid function:
$
\hat{p} = \frac{1}{1 + e^{-z}}.
$

In [ ]:
def predict_prob(model, X_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits)
    return probs.numpy().ravel()

val_prob_16 = predict_prob(model_16, X_val_tensor)
test_prob_16 = predict_prob(model_16, X_test_tensor)

print(val_prob_16[:10].round(4))

## 14. Evaluate the first neural network

We use:

- accuracy with a $0.5$ threshold
- ROC AUC for discrimination
- the confusion matrix on the test set



In [ ]:
val_pred_16 = (val_prob_16 >= 0.5).astype(int)
test_pred_16 = (test_prob_16 >= 0.5).astype(int)

nn16_results = pd.DataFrame({
    "set": ["validation", "test"],
    "accuracy": [
        accuracy_score(y_val, val_pred_16),
        accuracy_score(y_test, test_pred_16)
    ],
    "auc": [
        roc_auc_score(y_val, val_prob_16),
        roc_auc_score(y_test, test_prob_16)
    ]
})

nn16_results.round(4)

In [ ]:
print("Test confusion matrix")
print(confusion_matrix(y_test, test_pred_16))
print()
print("Test classification report")
print(classification_report(y_test, test_pred_16, digits=4))

## 15. Compare with logistic regression

A neural network is more flexible than logistic regression, but more flexibility does not automatically imply better performance. A baseline comparison is therefore important.

In [ ]:
logit_model = LogisticRegression(max_iter=5000, random_state=SEED)
logit_model.fit(X_train_scaled, y_train)

val_prob_logit = logit_model.predict_proba(X_val_scaled)[:, 1]
test_prob_logit = logit_model.predict_proba(X_test_scaled)[:, 1]

val_pred_logit = (val_prob_logit >= 0.5).astype(int)
test_pred_logit = (test_prob_logit >= 0.5).astype(int)

baseline_results = pd.DataFrame({
    "model": ["logistic regression", "neural network (16 units)"],
    "validation_accuracy": [
        accuracy_score(y_val, val_pred_logit),
        accuracy_score(y_val, val_pred_16)
    ],
    "validation_auc": [
        roc_auc_score(y_val, val_prob_logit),
        roc_auc_score(y_val, val_prob_16)
    ],
    "test_accuracy": [
        accuracy_score(y_test, test_pred_logit),
        accuracy_score(y_test, test_pred_16)
    ],
    "test_auc": [
        roc_auc_score(y_test, test_prob_logit),
        roc_auc_score(y_test, test_prob_16)
    ]
})

baseline_results.round(4)

## 13. Summary

In this lab, we moved from the structure of a neural network to the practical mechanics of training one.

We used a simple feedforward neural network for binary classification and trained it by minimizing binary cross-entropy loss. The training process relied on backpropagation, which uses the chain rule to determine how each weight should be updated.

The loss curves showed an important practical lesson. Although the training loss kept decreasing, the validation loss began to rise after the early epochs. This pattern indicates overfitting. It shows why neural network training should not be judged by training loss alone.

A validation set helps us monitor generalization and decide whether a model is being trained too long. In the next lab, we will build on this idea by comparing different neural network settings and using the validation set for model selection.

## 14. Additional practice

Try one or more of the following:

1. Change the learning rate from $0.01$ to $0.001$. Does the loss decrease more slowly?
2. Change the learning rate from $0.01$ to $0.1$. Does the training process become less stable?
3. Write one or two sentences explaining why a lower training loss does not necessarily mean a better model.
